In [96]:
import GtoTmodel as GtoTmodel
import torch
import torch.nn as nn
import torch.optim as optim

In [97]:
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 10  # Number of colomns in the graph
text_vocab_size = 26  # Vocabulary size for text

Importing the model

In [98]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [99]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [100]:
learning_rate = 0.001
num_epochs = 10000

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Sample trainig set

In [101]:
num_ciruits = 7
graph_dataset =  torch.randn(num_ciruits,graph_colomns, graph_input_dim )
text_dataset = torch.randint(0, text_vocab_size, (num_ciruits,graph_colomns))

graph_dataset = graph_dataset.to(device)
text_dataset = text_dataset.to(device)


In [102]:
graph_dataset[0]

tensor([[-0.7672, -0.2704, -1.0904,  1.3722, -0.0407, -0.5890,  0.4869, -0.9667,
          1.0371,  1.2833],
        [ 0.8447,  0.7073,  2.3386, -0.9650,  0.7166,  0.3069,  0.3666, -0.4037,
          1.4114,  0.0441],
        [ 2.0300,  0.6579,  0.6814, -2.2222, -0.2469, -0.7786,  0.0811,  0.2718,
         -0.3849,  1.1882],
        [ 0.7338,  2.4685, -1.1564,  1.0574,  0.2089,  0.8413, -1.0309, -0.5808,
         -0.2106, -0.9223],
        [ 1.2604, -1.0519,  0.3423, -0.1402, -0.3604, -0.0691, -0.8428,  1.5959,
         -0.4949,  0.4362]], device='cuda:0')

Split data in to test and train set

In [103]:
from sklearn.model_selection import train_test_split

graph_train, graph_test, text_train, text_test = train_test_split(graph_dataset, text_dataset, test_size=0.2)



In [105]:
# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    # Forward pass
    for i in range(0, len(graph_train), batch_size):
        graph_data = graph_train[i:i+batch_size]
        text = text_train[i:i+batch_size]
        text_input = text[:, :-1]
        print(text_input)
        target = text[:,-2:-1].reshape(-1)
        print(target)
        output = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
        output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation
        print(output.shape)
        print(output)
        # Compute loss
        loss = criterion(output, target)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:  # Print every 100 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

tensor([[13,  4,  2, 17],
        [ 2,  7,  5, 13],
        [14, 19, 19, 21],
        [22, 12, 19, 12],
        [20, 22,  5, 13]], device='cuda:0')
tensor([17, 13, 21, 12, 13], device='cuda:0')
torch.Size([20, 26])
tensor([[ 0.7640, -0.5154,  0.6071,  0.1000, -0.1391, -0.2955,  1.0836,  0.5318,
          0.0759, -0.5042,  0.7225, -0.6109,  0.0858, -0.6129, -0.5539, -0.8156,
         -1.3293,  0.3746, -0.4014,  0.0948,  0.1518,  0.4995,  1.2357,  0.3471,
         -0.4298, -0.5770],
        [ 0.7390, -0.2471,  0.3513, -0.0578, -1.4205, -0.5712,  0.2612,  0.5130,
          0.5990, -0.6161, -0.3369, -0.3434,  0.0125, -0.4500, -1.0562, -0.7868,
         -1.3925, -0.1790, -0.3261,  0.8019,  0.6040,  0.7749,  0.8650, -0.4275,
         -0.4430, -0.1640],
        [ 0.7973, -0.4258,  0.9227,  0.0985, -0.5121, -0.6605,  0.1223,  0.2053,
          0.3451, -0.0220,  0.4633, -0.0537, -0.0081, -0.8006, -0.2649,  0.2152,
         -1.4490,  0.0058, -0.8049,  0.2745,  0.2052,  0.5192,  0.9132, -0.1322,


ValueError: Expected input batch_size (20) to match target batch_size (5).

In [ ]:
# Forward pass
output = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation

# Compute loss
loss = criterion(output, target)
print(f"Loss: {loss.item():.4f}")